# Análise Exploratória: City_Tier × Rent
Investigando a relação entre o nível da cidade e o valor pago de aluguel.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data_3.csv')

# Ordenar tiers
tier_order = ['Tier_1', 'Tier_2', 'Tier_3']
df['City_Tier'] = pd.Categorical(df['City_Tier'], categories=tier_order, ordered=True)
df = df.sort_values('City_Tier')

print(f'Shape: {df.shape}')
print(f"\nDistribuição de City_Tier:\n{df['City_Tier'].value_counts().sort_index()}")
print(f"\nEstatísticas de Rent por City_Tier:")
df.groupby('City_Tier')['Rent'].describe().round(2)



Shape: (20000, 27)

Distribuição de City_Tier:
City_Tier
Tier_1     5934
Tier_2    10068
Tier_3     3998
Name: count, dtype: int64

Estatísticas de Rent por City_Tier:


,count,mean,std,min,25%,50%,75%,max
City_Tier,,,,,,,,
Tier_1,5934.0,12320.52,11656.71,411.53,5270.31,8927.18,15410.47,151482.29
Tier_2,10068.0,8342.64,8102.11,260.24,3532.74,6083.74,10374.25,215945.67
Tier_3,3998.0,6304.71,6067.87,235.37,2632.13,4523.14,7787.87,105418.93


## 1. Distribuição de registros por City_Tier

In [ ]:
contagem = df['City_Tier'].value_counts().sort_index().reset_index()
contagem.columns = ['City_Tier', 'Count']

fig = px.bar(
    contagem,
    x='City_Tier', y='Count',
    color='City_Tier',
    text='Count',
    title='Quantidade de Registros por City_Tier',
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

# Conta quantos imóveis existem em cada tier e plota um gráfico de barras

## 2. Box Plot — Distribuição do Rent por City_Tier

In [ ]:
fig = px.box(
    df, x='City_Tier', y='Rent',
    color='City_Tier',
    title='Distribuição do Aluguel por Nível de Cidade',
    labels={'Rent': 'Aluguel (R$)', 'City_Tier': 'Nível da Cidade'},
    color_discrete_sequence=px.colors.qualitative.Bold,
    points='outliers'
)
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

#Mostra a distribuição do aluguel por tier através de boxplots, com os outliers visíveis.
#O box plot resume 5 estatísticas ao mesmo tempo (mínimo, Q1, mediana, Q3, máximo) e expõe os valores atípicos. É ideal para comparar se 
# a centralidade e a dispersão diferem entre os grupos. Tier_1 tende a ter tanto maior mediana quanto mais outliers extremos.

## 3. Violin Plot — Densidade da distribuição

In [ ]:
fig = px.violin(
    df, x='City_Tier', y='Rent',
    color='City_Tier',
    box=True, points=False,
    title='Violin Plot — Aluguel por City_Tier',
    labels={'Rent': 'Aluguel (R$)', 'City_Tier': 'Nível da Cidade'},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

#Enquanto o box plot esconde se a distribuição é bimodal ou tem um "pico" específico, o violin revela isso. 
# Se um tier tem dois picos de frequência (por exemplo, imóveis baratos e caros sem meio-termo), o violin mostra — o box plot não.

## 4. Histograma por City_Tier

In [ ]:
fig = px.histogram(
    df, x='Rent', color='City_Tier',
    facet_col='City_Tier',
    nbins=60,
    title='Histograma do Aluguel por City_Tier',
    labels={'Rent': 'Aluguel (R$)'},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

#Exibe a frequência dos valores de aluguel em faixas (bins) para cada tier separadamente, usando facet_col.
#O histograma é mais direto para identificar assimetria (distribuição com cauda para a direita, o que é comum em preços de aluguel), concentrações de valores e a 
# presença de múltiplos grupos dentro de um tier.

## 5. Estatísticas resumidas — Média, Mediana e Desvio Padrão

In [ ]:
stats = df.groupby('City_Tier')['Rent'].agg(
    Media='mean', Mediana='median', Desvio_Padrao='std',
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75)
).reset_index().round(2)

fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Média do Aluguel', 'Mediana do Aluguel', 'Desvio Padrão'])

colors = px.colors.qualitative.Bold[:3]

for i, (col, titulo) in enumerate(zip(['Media', 'Mediana', 'Desvio_Padrao'], ['Média', 'Mediana', 'Desvio Padrão']), 1):
    fig.add_trace(
        go.Bar(x=stats['City_Tier'], y=stats[col],
               marker_color=colors,
               text=stats[col], textposition='outside',
               showlegend=False),
        row=1, col=i
    )

fig.update_layout(title='Estatísticas do Aluguel por City_Tier', plot_bgcolor='white', height=450)
fig.show()
print(stats.to_string(index=False))

#Calcula média, mediana, desvio padrão, Q1 e Q3 por tier, e plota os três primeiros lado a lado em barras.
#Se a média for bem maior que a mediana, significa que outliers de alto valor estão puxando a média para cima — o que distorceria conclusões se usássemos só a média. 
# O desvio padrão alto em todos os tiers indica que "nível da cidade" sozinho não explica toda a variação do aluguel.

City_Tier    Media  Mediana  Desvio_Padrao      Q1       Q3
   Tier_1 12320.52  8927.18       11656.71 5270.31 15410.47
   Tier_2  8342.64  6083.74        8102.11 3532.74 10374.25
   Tier_3  6304.71  4523.14        6067.87 2632.13  7787.87


## 6. Proporção do Rent em relação à Renda (Rent/Income) por City_Tier

In [ ]:
df['Rent_Income_Ratio'] = (df['Rent'] / df['Income']) * 100

fig = px.box(
    df, x='City_Tier', y='Rent_Income_Ratio',
    color='City_Tier',
    title='% da Renda Gasta com Aluguel por City_Tier',
    labels={'Rent_Income_Ratio': '% Renda → Aluguel', 'City_Tier': 'Nível da Cidade'},
    color_discrete_sequence=px.colors.qualitative.Bold,
    points='outliers'
)
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

print("Média do % de renda gasto com aluguel por City_Tier:")
print(df.groupby('City_Tier')['Rent_Income_Ratio'].mean().round(2))

#Cria uma variável derivada (Rent_Income_Ratio) que expressa o aluguel como percentual da renda, e plota um box plot dessa proporção por tier.
#Essa é talvez a seção mais analítica do notebook. Comparar o valor absoluto do aluguel entre tiers pode ser enganoso — faz mais sentido perguntar: em qual tier as pessoas comprometem mais renda com moradia? A razão normaliza por renda e responde isso diretamente. 
# Aqui sim é um indicador de peso financeiro relativo do aluguel.

Média do % de renda gasto com aluguel por City_Tier:
City_Tier
Tier_1    30.0
Tier_2    20.0
Tier_3    15.0
Name: Rent_Income_Ratio, dtype: float64


## 7. Conclusões

Com base na análise acima, podemos concluir:

- **Tier_1** apresenta os maiores valores médios e medianos de aluguel, seguido por Tier_2 e Tier_3 — o que é esperado, pois cidades maiores tendem a ter custo de vida mais elevado.
- **Tier_2** concentra a maior parte dos registros (~50%), sendo o grupo mais representativo do dataset.
- O **desvio padrão** é alto em todos os tiers, indicando grande variabilidade nos valores de aluguel mesmo dentro de um mesmo nível de cidade.
- A **relação Rent/Income** indica quanto da renda é comprometida com aluguel — cidades Tier_1 tendem a ter essa proporção mais alta.
